End-to-End Cleaning Workflow + Mini-Project

Today everything comes together. NeoMart wants to predict which customers might leave (a "churn"-style problem). The CRM export is messy: inconsistent names, a missing plan, a missing spend, dates stored as text, amounts stored as text with commas, and a duplicated row. Our job: turn it into a clean, fully numeric table a machine-learning model can consume. Run the three parts in order — Part 1 creates the file the later parts read (like the Module 8 pipeline).

In [ ]:
import pandas as pd

In [9]:
# Part 1: a messy export from NeoMart's CRM system
import pandas as pd

# Part 1: a messy export from NeoMart's CRM system
raw = """cust_id,name,plan,signup_date,monthly_spend
1,priya sharma,Premium,2024-03-15,"2,450.50"
2,rahul verma,Basic,2025-01-10,1200.00
3,anita desai,,2023-11-02,"3,100.00"
4,amit joshi,Basic,2026-02-20,
2,rahul verma,Basic,2025-01-10,1200.00
5,sneha rao,Premium,2025-07-01,2890.75
"""
with open("customers_raw.csv", "w") as f:
    f.write(raw)
df = pd.read_csv("customers_raw.csv")
print(df)
print(df.isna().sum())
print("Duplicate Rows :", df.duplicated())
print(df.dtypes)


   cust_id          name     plan signup_date monthly_spend
0        1  priya sharma  Premium  2024-03-15      2,450.50
1        2   rahul verma    Basic  2025-01-10       1200.00
2        3   anita desai      NaN  2023-11-02      3,100.00
3        4    amit joshi    Basic  2026-02-20           NaN
4        2   rahul verma    Basic  2025-01-10       1200.00
5        5     sneha rao  Premium  2025-07-01       2890.75
cust_id          0
name             0
plan             1
signup_date      0
monthly_spend    1
dtype: int64
Duplicate Rows : 0    False
1    False
2    False
3    False
4     True
5    False
dtype: bool
cust_id          int64
name               str
plan               str
signup_date        str
monthly_spend      str
dtype: object


Part 2: clean

In [16]:
import pandas as pd
df = pd.read_csv("customers_raw.csv")
df = df.drop_duplicates()
df["name"] = df["name"].str.strip().str.title()
df["monthly_spend"] = (df["monthly_spend"]
                       .str.replace(",","")
                       .astype(float)
)
df["monthly_spend"] = df["monthly_spend"].fillna(df["monthly_spend"].mean())
df["plan"] = df["plan"].fillna("Basic")
df["signup_date"] = pd.to_datetime(df["signup_date"])
df.to_csv("customers_raw.csv", index=False)
print(df)

   cust_id          name     plan signup_date  monthly_spend
0        1  Priya Sharma  Premium  2024-03-15      2450.5000
1        2   Rahul Verma    Basic  2025-01-10      1200.0000
2        3   Anita Desai    Basic  2023-11-02      3100.0000
3        4    Amit Joshi    Basic  2026-02-20      2410.3125
5        5     Sneha Rao  Premium  2025-07-01      2890.7500


Part 3: engineer features, encode, export

Two simple features: tenure_days (how long a customer has been with NeoMart — loyal customers behave differently) and is_high_value (spend above Rs 2,500). Then pd.get_dummies one-hot encodes the categorical plan column into 0/1 columns, because models cannot read the word "Premium".

In [20]:
import pandas as pd

# Part 3: features + encoding (reads the file Part 2 created)
df = pd.read_csv("customers_raw.csv", parse_dates=["signup_date"])
snapshot = pd.Timestamp("2026-08-12")  

df["tenure_days"] = (snapshot - df["signup_date"]).dt.days
df["is_high_value"] = (df["monthly_spend"] > 2500).astype(int)
df = pd.get_dummies(df, columns=["plan"], dtype=int)   # one-hot encode
df = df.drop(columns=["name", "signup_date"]) 

df.to_csv("customers_ml_ready.csv", index= False)
print(df.head())
print(df.dtypes)

   cust_id  monthly_spend  tenure_days  is_high_value  plan_Basic  \
0        1      2450.5000          880              0           0   
1        2      1200.0000          579              0           1   
2        3      3100.0000         1014              1           1   
3        4      2410.3125          173              0           1   
4        5      2890.7500          407              1           0   

   plan_Premium  
0             1  
1             0  
2             0  
3             0  
4             1  
cust_id            int64
monthly_spend    float64
tenure_days        int64
is_high_value      int64
plan_Basic         int64
plan_Premium       int64
dtype: object


Subtracting datetimes gives a duration; .dt.days extracts whole days — that is tenure_days.
dtype=int makes the dummy columns 0/1 (the pandas 3.x default is True/False booleans).
Every column is now numeric: this table is ready for Machine Learning — exactly what you will feed to scikit-learn in your ML course. The customers_ml_ready.csv file is the hand-off artifact.

P1 — First table (Series + DataFrame warm-up)

Make a Series of a gym's member check-ins for Mon–Fri: 42, 38, 51, 47, 60, with day-name labels; print the busiest day's count and the weekly average.
Build a DataFrame of 3 courses (course, fee): Python 15000, SQL 9000, Statistics 12000; print it and its shape.

In [24]:
import pandas as pd
checkins = pd.Series([42, 38, 51, 47, 60],
                      index=["Mon", "Tue", "wed", "Thu", "Fri"])
print("Busiest_Day:", checkins.max())
print("Avarege:", checkins.mean())

courses = pd.DataFrame({
    'course':["python", "Sql", "Statistics"],
    "fees": [15000, 9000, 12000]
})
print(courses)
print(courses.shape)

Busiest_Day: 60
Avarege: 47.6
       course   fees
0      python  15000
1         Sql   9000
2  Statistics  12000
(3, 2)


AP2 — Filter and sort drill

MediCare appointments: departments ["Cardiology", "General", "Cardiology", "Dermatology", "General"], fees [1250.0, 600.0, 1450.0, 900.0, 550.0]. Build the DataFrame, then print: all Cardiology rows · rows with fee ≥ 900 sorted largest-first · the total fees of that filtered set.

In [28]:
Medicare = pd.DataFrame({
    "department" :["Cardiology", "General", "Cardiology", "Dermatology", "General"],
    "fees": [1200.0, 600.0,1450.0, 900.0, 550.0 ]
})
print(Medicare[Medicare["department"] ==  "Cardiology"])
big = (Medicare[Medicare['fees'] >= 900].sort_values("fees", ascending=False))
print(big)
print("Total of fees >= 900:", big["fees"].sum())


   department    fees
0  Cardiology  1200.0
2  Cardiology  1450.0
    department    fees
2   Cardiology  1450.0
0   Cardiology  1200.0
3  Dermatology   900.0
Total of fees >= 900: 3550.0


AP3 — Cleaning drill (.str, astype, dates)

A messy export: names [" divya menon", "ARJUN NAIR "], amounts as text ["1,499.50", "2,999.00"], order dates ["2026-08-10", "2026-08-15"]. Clean the names (strip + Title Case), convert amounts to float (comma-strip first!), parse the dates and add a weekday column. Print the final table and its dtypes.

In [42]:
import pandas as pd

df = pd.DataFrame({
    "name": ["  divya menon", "ARJUN NAIR "],
    "amount": ["1,499.50", "2,999.00"],
    "order_date": ["2026-08-10", "2026-08-15"]
})
df["name"] = df["name"].str.strip().str.title
df["amount"] = df["amount"].str.replace(",", "").astype(float)
df["order_date"] = pd.to_datetime(df["order_date"])
df["weekday"] = df["order_date"].dt.day_name()
print(df)
print(df.dtypes)

                                                name  amount order_date  \
0  <bound method StringMethods.title of <pandas.c...  1499.5 2026-08-10   
1  <bound method StringMethods.title of <pandas.c...  2999.0 2026-08-15   

    weekday  
0    Monday  
1  Saturday  
name                  object
amount               float64
order_date    datetime64[us]
weekday                  str
dtype: object


AP5 — Challenge: two-table mini pipeline (merge → quality check → clean → summarise)

SafeBank has an accounts table — acc_no ["SB201", "SB202", "SB203"], holder ["Divya Menon", "Arjun Nair", "Meera Iyer"], branch ["Andheri", "Pune", "Andheri"] — and a transactions table — acc_no ["SB201", "SB203", "SB201", "SB209"], amount [5000.0, 1200.0, 800.0, 999.0].

1. Left-merge transactions with accounts and print the result.
2. Data-quality check: print how many transactions found no account (holder   column's isna().sum()).
3. Fill the missing holder with "UNKNOWN" and the missing branch with "Unassigned" (assign back — Copy-on-Write!).
4. Print total amount per branch, sorted largest first.

In [56]:
accounts = pd.DataFrame({
    "acc_no":["SB201", "SB202", "SB203"],
    "holder" : ["Divya Menon", "Arjun Nair", "Meera Iyer"],
    "branch" : ["Andheri", "Pune", "Andheri"],
})
transactions = pd.DataFrame({
    "acc_no" : ["SB201", "SB203", "SB201", "SB209"],
    "amount" : [5000.0, 1200.0, 800.0, 999.0]
})
# 1. Left merge keeps every transaction
Details = pd.merge(transactions, accounts, on="acc_no", how="left")
print(Details)

# 2. Quality check: unmatched transactions
print("Transactions with no account:", Details["holder"].isna().sum())

# 3. Fill the gaps (assign back - pandas 3.x Copy-on-Write)
Details["holder"] = Details["holder"].fillna("UNKNOWN")
Details["branch"] = Details["branch"].fillna("Unassigned")

# 4. Total per branch, largest first

print(Details.groupby("branch")["amount"].sum().sort_values(ascending=False))

  acc_no  amount       holder   branch
0  SB201  5000.0  Divya Menon  Andheri
1  SB203  1200.0   Meera Iyer  Andheri
2  SB201   800.0  Divya Menon  Andheri
3  SB209   999.0          NaN      NaN
Transactions with no account: 1
branch
Andheri       7000.0
Unassigned     999.0
Name: amount, dtype: float64
